# MAI-Image-2: State-of-the-Art Text-to-Image Generation

> **Model Card:** [ai.azure.com/catalog/models/MAI-Image-2](https://ai.azure.com/catalog/models/MAI-Image-2)

MAI-Image-2 is Microsoft AI's highest-capability text-to-image model — debuting **#3 on the Arena.ai leaderboard** for image model families. Developed in collaboration with photographers, designers, and visual storytellers, it powers image generation in Copilot, Bing Image Creator, and PowerPoint.

| Attribute | MAI-Image-2 | MAI-Image-2e (Efficient) |
|---|---|---|
| **Model type** | Text-to-image (diffusion) | Text-to-image (diffusion, optimized) |
| **Best for** | Maximum fidelity, complex scenes | High-volume, fast turnaround |
| **Speed** | High quality | 22% faster than MAI-Image-2 |
| **Efficiency** | — | 4× more efficient |
| **Input** | Text prompt (≤32,000 tokens) | Text prompt (≤32,000 tokens) |
| **Output** | PNG image | PNG image |
| **Dimensions** | Min 768×768, max 1024×1024\* | Min 768×768, max 1024×1024\* |
| **Regions** | West Central US · East US · West US · West Europe · Sweden Central · South India | Same |
| **Pricing (input)** | **\$5 / 1M text tokens** | **\$5 / 1M text tokens** |
| **Pricing (output)** | **\$33 / 1M image tokens** | Lower (efficient variant) |

\* Total pixel count must not exceed 1,048,576 (≈ 1024×1024). Either dimension can exceed 1024 as long as total stays within limit.

### Capabilities
- **Photorealistic image synthesis** — consistent visual structure, natural lighting
- **Superior text rendering** — headlines, labels, infographics
- **Creative range** — anime, illustration, cinematic, product shots
- **Complex layouts** — multi-subject scenes, precise composition

## 1. Setup

In [ ]:
%pip install -q requests python-dotenv Pillow azure-identity

In [ ]:
import os
import base64
import json
import time
import requests
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Image as IPImage, display
from PIL import Image
import io

load_dotenv(override=True)

AZURE_FOUNDRY_ENDPOINT = os.getenv("AZURE_FOUNDRY_ENDPOINT")
AZURE_FOUNDRY_API_KEY  = os.getenv("AZURE_FOUNDRY_API_KEY")
SHARED_IMAGE_DEPLOYMENT = os.getenv("MAI_IMAGE_2E_DEPLOYMENT_NAME") or os.getenv("MAI_IMAGE_2_DEPLOYMENT_NAME", "mai-image-2e")

# Keep both aliases for backwards compatibility in later notebook cells.
DEPLOYMENT_MAI_IMAGE_2  = SHARED_IMAGE_DEPLOYMENT
DEPLOYMENT_MAI_IMAGE_2E = SHARED_IMAGE_DEPLOYMENT

assert AZURE_FOUNDRY_ENDPOINT, "Set AZURE_FOUNDRY_ENDPOINT in your .env file"
assert AZURE_FOUNDRY_API_KEY,  "Set AZURE_FOUNDRY_API_KEY in your .env file"

IMAGES_API_URL = f"{AZURE_FOUNDRY_ENDPOINT.rstrip('/')}/mai/v1/images/generations"

print(f"✅ Images API URL      : {IMAGES_API_URL}")
print(f"✅ Shared deployment   : {SHARED_IMAGE_DEPLOYMENT}")
print(f"✅ MAI-Image-2 alias   : {DEPLOYMENT_MAI_IMAGE_2}")
print(f"✅ MAI-Image-2e alias  : {DEPLOYMENT_MAI_IMAGE_2E}")

## 2. Core Helper Function

In [ ]:
def generate_image(
    prompt: str,
    output_path: str,
    deployment: str = DEPLOYMENT_MAI_IMAGE_2E,
    width: int = 1024,
    height: int = 1024,
) -> dict:
    """
    Generate an image with MAI-Image-2 or MAI-Image-2e.
    Returns the raw API response dict.
    Constraints: width >= 768, height >= 768, width*height <= 1,048,576
    """
    assert width >= 768 and height >= 768, "Both width and height must be ≥ 768"
    assert width * height <= 1_048_576,    "width × height must not exceed 1,048,576"

    payload = {
        "model":  deployment,
        "prompt": prompt,
        "width":  width,
        "height": height,
    }

    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_FOUNDRY_API_KEY,
    }

    start    = time.time()
    response = requests.post(IMAGES_API_URL, headers=headers, json=payload)
    elapsed  = time.time() - start

    response.raise_for_status()
    result = response.json()

    # Decode and save image
    image_data = [item for item in result.get("data", []) if "b64_json" in item]
    if image_data:
        img_bytes = base64.b64decode(image_data[0]["b64_json"])
        with open(output_path, "wb") as f:
            f.write(img_bytes)
        size_kb = len(img_bytes) / 1024
        print(f"✅ Saved: {output_path} ({size_kb:.0f} KB)  |  {width}×{height}  |  {elapsed:.1f}s")
    else:
        print("⚠️ No image data in response:", result)

    return result


def show_image(path: str, title: str = "") -> None:
    """Display a saved image inline in Jupyter."""
    img = Image.open(path)
    print(f"{title} ({img.size[0]}×{img.size[1]})")
    display(IPImage(filename=path, width=512))

## 3. Basic Image Generation

In [ ]:
result = generate_image(
    prompt="A photorealistic mountain lake at sunrise, misty atmosphere, "
           "golden hour lighting, crystal clear water reflecting snow-capped peaks",
    output_path="img_landscape.png",
)
show_image("img_landscape.png", "Photorealistic Landscape")

In [ ]:
# Product shot: clean studio style
generate_image(
    prompt="Professional product photography of a sleek midnight-blue wireless headphone "
           "on a white marble surface, soft studio lighting, subtle shadow",
    output_path="img_product.png",
)
show_image("img_product.png", "Product Photography")

## 4. Text Rendering in Images

MAI-Image-2 excels at in-image text for infographics, banners, and diagrams.

In [ ]:
generate_image(
    prompt="A modern tech conference banner with large bold text: 'Microsoft Foundry 2026' "
           "on a deep blue gradient background with subtle circuit-board patterns, "
           "professional design, clean typography",
    output_path="img_banner.png",
)
show_image("img_banner.png", "Conference Banner with Text")

In [ ]:
# Infographic with data labels
generate_image(
    prompt="A clean flat-style infographic showing three icons side by side: "
           "a microphone labeled 'Transcribe', a speaker labeled 'Voice', "
           "and an image frame labeled 'Image'. Minimal white background, "
           "blue and orange accent colors, modern sans-serif font",
    output_path="img_infographic.png",
)
show_image("img_infographic.png", "Infographic with Labels")

## 5. Creative & Stylized Outputs

In [ ]:
# Anime/illustration style
generate_image(
    prompt="Anime-style illustration of a futuristic AI assistant floating above a city "
           "at night, glowing teal circuits, vibrant neon colors, detailed background, "
           "Studio Ghibli aesthetic",
    output_path="img_anime.png",
)
show_image("img_anime.png", "Anime Style")

In [ ]:
# Cinematic portrait
generate_image(
    prompt="Cinematic portrait of a software engineer at a futuristic holographic workstation, "
           "dramatic side lighting, shallow depth of field, photorealistic, "
           "8K resolution quality, cinematic color grading",
    output_path="img_portrait.png",
)
show_image("img_portrait.png", "Cinematic Portrait")

## 6. Unified Deployment (MAI-Image-2e)

This notebook now uses a single shared deployment for both aliases (`DEPLOYMENT_MAI_IMAGE_2` and `DEPLOYMENT_MAI_IMAGE_2E`).

Use this section to validate output quality and latency from the shared endpoint in a production-like flow.

In [ ]:
import time

prompt = (
    "Professional product shot of a smartphone with a vibrant abstract wallpaper, "
    "white background, clean studio lighting"
)

print(f"Using shared deployment: {DEPLOYMENT_MAI_IMAGE_2E}")
t0 = time.time()
generate_image(prompt, "img_unified_eff.png", deployment=DEPLOYMENT_MAI_IMAGE_2E)
latency = time.time() - t0

print(f"\n{'Deployment':<28} {'Latency':>10}")
print("-" * 42)
print(f"{DEPLOYMENT_MAI_IMAGE_2E:<28} {latency:>9.2f}s")

show_image("img_unified_eff.png", "Shared Deployment Output")

In [ ]:
# Optional second render from the same shared deployment (useful for prompt iteration)
variant_prompt = (
    "Professional product shot of a smartphone with a vibrant abstract wallpaper, "
    "dark graphite background, dramatic rim lighting, premium advertising style"
 )

generate_image(
    variant_prompt,
    "img_unified_eff_variant.png",
    deployment=DEPLOYMENT_MAI_IMAGE_2E,
 )
show_image("img_unified_eff_variant.png", "Variant Render (same deployment)")

## 7. Custom Dimensions

Generate landscape, portrait, or square images. Constraint: `width × height ≤ 1,048,576`, both ≥ 768.

In [ ]:
dimension_configs = [
    ("Square 1024×1024",   1024, 1024),
    ("Landscape 1024×768", 1024, 768),
    ("Portrait 768×1024",  768,  1024),
    ("Wide 1365×768",      1365, 768),   # 1365*768 = 1,048,320 ✅ just under limit
]

prompt_dim = "A serene Japanese zen garden with raked gravel, moss-covered stones, and a single cherry blossom tree"

for label, w, h in dimension_configs:
    out = f"img_dim_{w}x{h}.png"
    print(f"\n{label} ({w*h:,} total pixels)")
    generate_image(prompt_dim, out, deployment=DEPLOYMENT_MAI_IMAGE_2E, width=w, height=h)
    show_image(out, label)

## 8. Batch Generation Pipeline

Generate multiple images from a list of prompts — useful for marketing campaigns or content pipelines.

In [ ]:
import concurrent.futures

BATCH_PROMPTS = [
    ("A warm cozy café interior with exposed brick, morning light through tall windows", "batch_cafe.png"),
    ("Futuristic electric sports car on a coastal highway at dusk, dramatic sky",       "batch_car.png"),
    ("Overhead flat-lay of a healthy meal prep with colorful vegetables and grains",    "batch_food.png"),
    ("Abstract geometric art in navy, gold, and cream, suitable for office wall decor", "batch_art.png"),
]

def generate_batch_item(args):
    prompt, output_path = args
    try:
        generate_image(prompt, output_path, deployment=DEPLOYMENT_MAI_IMAGE_2E)
        return output_path, None
    except Exception as e:
        return output_path, str(e)

print(f"Generating {len(BATCH_PROMPTS)} images (using MAI-Image-2e for cost efficiency)...")
t_start = time.time()

# Sequential generation (rate limit friendly)
results = []
for args in BATCH_PROMPTS:
    path, err = generate_batch_item(args)
    results.append((path, err))

total_time = time.time() - t_start
succeeded  = sum(1 for _, e in results if e is None)
print(f"\n✅ {succeeded}/{len(BATCH_PROMPTS)} images generated in {total_time:.1f}s")

# Display all
for path, err in results:
    if err is None:
        show_image(path, path)

## 9. Microsoft Entra ID Authentication (Production)

In [ ]:
# Uncomment to use managed identity / Entra ID instead of API keys

# from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# token_provider = get_bearer_token_provider(
#     DefaultAzureCredential(),
#     "https://cognitiveservices.azure.com/.default"
# )
# entra_token = token_provider()

# entra_headers = {
#     "Content-Type": "application/json",
#     "Authorization": f"Bearer {entra_token}",
# }
# # Use entra_headers in requests.post() instead of api-key header

print("Entra ID auth snippet above — uncomment to use managed identity.")

## 10. 💰 Cost Calculator

**MAI-Image-2 pricing:**
- Text input: **\$5 per 1M tokens**
- Image output: **\$33 per 1M image tokens**

**MAI-Image-2e** (efficient) is approximately 4× more cost-efficient for image output.

**Token estimation:**
- Text input: ~1 token per 4 chars (standard GPT tokenization). A 100-word prompt ≈ 133 tokens.
- Image output tokens: each image is counted as a fixed number of tokens based on resolution.

In [ ]:
# ── Cost Calculator ─────────────────────────────────────────
PRICE_TEXT_PER_1M   = 5.00   # USD per 1M text input tokens
PRICE_IMAGE_PER_1M  = 33.00  # USD per 1M image output tokens (MAI-Image-2)

# Approximate image output tokens per image (based on Azure image token counting):
# 1024×1024 ~ 1,056 tokens (approximate; consult Azure pricing page for exact)
IMAGE_TOKENS_PER_IMAGE = 1056  # approximate for 1024×1024

# Approximate text tokens for a typical prompt
AVG_PROMPT_TOKENS = 75  # ~75 tokens per prompt (300 chars)

scenarios = {
    "Prototype / 10 images":          10,
    "Daily marketing batch (100)":    100,
    "Weekly pipeline (1,000)":      1_000,
    "Monthly production (10,000)": 10_000,
    "Enterprise scale (100,000)": 100_000,
}

print(f"\nMAI-Image-2 Cost Estimator")
print(f"  Text input : ${PRICE_TEXT_PER_1M}/1M tokens")
print(f"  Image output: ${PRICE_IMAGE_PER_1M}/1M tokens (~{IMAGE_TOKENS_PER_IMAGE} tokens/image at 1024×1024)")
print()
print(f"{'Scenario':<35} {'Images':>8} {'Text cost':>12} {'Image cost':>12} {'Total':>12}")
print("-" * 81)

for label, n_images in scenarios.items():
    text_cost  = (n_images * AVG_PROMPT_TOKENS / 1_000_000) * PRICE_TEXT_PER_1M
    image_cost = (n_images * IMAGE_TOKENS_PER_IMAGE / 1_000_000) * PRICE_IMAGE_PER_1M
    total      = text_cost + image_cost
    print(f"{label:<35} {n_images:>8,} ${text_cost:>11.4f} ${image_cost:>11.4f} ${total:>11.4f}")

print()
print("Note: MAI-Image-2e is 4× more efficient — image output cost ~75% lower.")
print(f"{'Scenario':<35} {'Images':>8} {'Image cost (2e)':>16} {'Total (2e)':>12}")
print("-" * 73)
for label, n_images in scenarios.items():
    text_cost   = (n_images * AVG_PROMPT_TOKENS / 1_000_000) * PRICE_TEXT_PER_1M
    image_cost_e = (n_images * IMAGE_TOKENS_PER_IMAGE / 1_000_000) * PRICE_IMAGE_PER_1M * 0.25  # ~4× cheaper
    total_e      = text_cost + image_cost_e
    print(f"{label:<35} {n_images:>8,} ${image_cost_e:>15.4f} ${total_e:>11.4f}")

## 11. Summary & Next Steps

| Feature | MAI-Image-2 | MAI-Image-2e |
|---|---|---|
| Photorealistic synthesis | ✅ | ✅ |
| In-image text rendering | ✅ | ✅ (short text) |
| Complex multi-subject layouts | ✅ | ✅ |
| Anime / illustration styles | ✅ | ✅ |
| Batch pipeline | ✅ | ✅ (recommended) |
| Custom dimensions | ✅ | ✅ |
| Speed | High | 22% faster |
| Efficiency | — | 4× more efficient |

**Resources:**
- [Model Card: MAI-Image-2](https://ai.azure.com/catalog/models/MAI-Image-2)
- [MS Docs: Use Foundry MAI Models](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-mai)
- [Deploy MAI-Image-2 in Foundry portal](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/deploy-foundry-models)
- [Arena.ai leaderboard](https://arena.ai/leaderboard/text-to-image)
- [MAI Playground](https://playground.microsoft.ai)
- [Pricing details](https://azure.microsoft.com/pricing/details/cognitive-services)